Para começar:

In [1]:
%pip install requests pandas numpy

  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Patent Extraction Pipeline
This section connects to the European Patent Office (EPO) to download patent data. It is designed to run smoothly and reliably using two main features:

1. **Auto-Authentication & Rate Limiting**: The script automatically refreshes your API token when it expires. It also adds small, controlled delays between requests so the EPO server doesn't block you for downloading too fast.

1. **Bypassing the 2,000-Result Limit (Time-Slicing)**: The EPO API only allows you to download 2,000 results at a time. If a single year has more than 2,000 patents, the script automatically breaks the search down month-by-month to ensure no data is left behind.

> **Note**: Keep your MY_KEY and MY_SECRET credentials private. If the EPO server gets overloaded with requests, the script is built to automatically pause and retry on its own.

In [ ]:
# 1. Importa as ferramentas de Extração (API)
from epo_api_v2 import check_epo_quota, download_raw_patents
import pandas as pd
from IPython.display import display

# 2. As tuas Credenciais da EPO API (NUNCA partilhes as chaves reais!)
MY_KEY = "your_key"
MY_SECRET = "your_secret"

# 3. Verifica a Quota e o Servidor primeiro
is_safe_to_run, current_quota_pct = check_epo_quota(MY_KEY, MY_SECRET)

# 4. Execução: DOWNLOAD
if is_safe_to_run:
    print("\n[SYSTEM] Beginning data extraction pipeline...")
    print("\n>>> STEP 1: DOWNLOADING RAW DATA <<<")
    
    # Faz o download e o teu script (epo_api.py) trata de guardar o CSV automaticamente
    raw_df = download_raw_patents(
        consumer_key=MY_KEY, 
        consumer_secret=MY_SECRET, 
        start_year=2021,
        end_year=2021
        #, applicant_filter="Alnylam" # Descomenta se quiseres filtrar já aqui
    )
        
    if not raw_df.empty:
        print(f"\n[SUCCESS] Extração concluída.")
        display(raw_df.head(10)) 
    else:
        print("\n[INFO] A pesquisa não encontrou resultados ou ocorreu um erro na extração.")
    
else:
    print(f"\n[SYSTEM] Pipeline abortado. Quota atual: {current_quota_pct}%. Verifique a ligação ou os limites da API.")

In [ ]:
# 1. Importa as ferramentas necessárias
import pandas as pd
from epo_filter_v2 import apply_filters
from IPython.display import display

print("\n[SYSTEM] Beginning local filter testing...")

# 2. Define o nome do ficheiro raw que já descarregaste antes
raw_filename = 'EPO_siRNA_RAW_2021_2021.csv' # Coloca aqui o nome exato do teu ficheiro raw

try:
    # 3. Lê o CSV (Atenção ao sep=';' que definiste no epo_api.py)
    print(f"\n>>> LENDO DADOS RAW: {raw_filename} <<<")
    raw_df = pd.read_csv(raw_filename, sep=';', encoding='utf-8-sig')
    
    print(f"[OK] Ficheiro carregado! Total de patentes: {len(raw_df)}")
    display(raw_df.head(3))
    
    # 4. Aplica o teu filtro atual e dá um nome diferente para testares as versões
    output_test_name = 'TESTE_FILTRO_V1_2021_geral.csv'
    
    print("\n>>> APLICANDO FILTRO <<<")
    final_df = apply_filters(raw_df, output_test_name)
    
    print("\n[SYSTEM] Teste concluído com sucesso!")
    display(final_df.head(5))

except FileNotFoundError:
    print(f"\n[ERRO] O ficheiro '{raw_filename}' não foi encontrado.")
    print("Verifica se o nome está correto e se está na mesma pasta do Jupyter.")


[SYSTEM] Beginning local filter testing...

>>> LENDO DADOS RAW: EPO_siRNA_RAW_2021_2021.csv <<<
[OK] Ficheiro carregado! Total de patentes: 16484


,Patent_ID,Country,Number,Kind,Family_ID,Priority_Date,Publication_Date,Applicant,Title,Abstract,IPCs,CPCs
0,TW202140509A,TW,202140509,A,074004172,NaN,NaN,NaN,No title,本揭露係關於靶向人類染色體9開讀框72(C9orf72)基因之雙股核酸(dsRNAi)劑及組...,NaN,NaN
1,TW202100545A,TW,202100545,A,072337709,20210101.0,20210101.0,UNIV HEALTH NETWORK [CA],T cell receptors and methods of use thereof,The present disclosure is directed recombinant...,NaN,"A61K35/17, A61K38/20, A61K40/11, A61K40/32, A6..."
2,TW202100564A,TW,202100564,A,073290271,20210101.0,20210101.0,PROGEN CO LTD [KR] | PROGEN CO LTD [KR],A novel modified immunoglobulin Fc fusion prot...,The present invention relates to a novel immun...,NaN,"A61K38/00, A61P37/08, C07K14/46, C07K14/5428, ..."



>>> APLICANDO FILTRO <<<

=== STARTING LOCAL CLASSIFICATION ===
[INFO] No patents will be deleted — all go to CSV for manual curation.


c:\Users\filip\Desktop\Codigo\epo_filter.py:89: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_sirna_text = df['Search_Text'].str.contains(
c:\Users\filip\Desktop\Codigo\epo_filter.py:106: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_forbidden = df['Search_Text'].str.contains(
c:\Users\filip\Desktop\Codigo\epo_filter.py:117: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_diag = df['Search_Text'].str.contains(diagnostic_only, case=False, na=False, regex=True)
c:\Users\filip\Desktop\Codigo\epo_filter.py:118: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_no_treatment = ~df['Search_Text'].str.contains(
c:\Users\filip\Desk


[SUCCESS] Classification complete.
  Total input patents: 16484

  TIER 1 — siRNA Confirmed (Text + CPC)           474  ████
  TIER 2 — siRNA Confirmed (Text only)            219  ██
  TIER 3 — siRNA by CPC only (Check Abstract)    2689  ██████████████████████████
  TIER 4A — Mixed Tech (siRNA + Forbidden Term)   194  █
  TIER 4B — Diagnostic/Biomarker only (Review)   1305  █████████████
  TIER 5 — Agri/Vet with siRNA CPC (Review)       195  █
  TIER 6 — No siRNA Signal (Likely Irrelevant)   9450  ██████████████████████████████████████████████████████████████████████████████████████████████
  TIER 7 — Agri/Vet without siRNA (Likely Irrel   849  ████████

  [!] 127 patents flagged as Needs_Espacenet_Review (Unreadable text/abstract)

  Saved to: TESTE_FILTRO_V1_2021_geral.csv

[SYSTEM] Teste concluído com sucesso!


,Patent_ID,Priority_Date,Publication_Date,Applicant,Title,Abstract,Espacenet_Link,Tier,Needs_Espacenet_Review,IPCs,CPCs,Family_ID
0,--- TIER 1 — SIRNA CONFIRMED (TEXT + CPC) (474...,,,,,,,,,,,
1,CN112159807A,20210101.0,20210101.0,BIONEER CORP | 柏业公司,Novel double-stranded oligo RNA and pharmaceut...,The present invention relates to a novel siRNA...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),False,NaN,"A61K48/00, A61P1/16, A61P11/00, A61P11/02, A61...",054241409
2,CN112159812A,20210101.0,20210101.0,INST BIOPHYSICS CAS | 中国科学院生物物理研究所,Application of mitochondrial RNAi in silencing...,The invention provides a method for silencing ...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),False,NaN,"A61K31/713, A61P25/00, C12N15/113",073866847
3,CN112175947A,20210105.0,20210105.0,UNIV JINAN | 暨南大学,SiRNA for targeted inhibition of SOS1 gene exp...,The invention discloses siRNA for targeted inh...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),False,NaN,"A61K31/713, A61P35/02, C12N15/113",073925297
4,CN112175945A,20210105.0,20210105.0,SUZHOU BIOSYNTECH BIOTECHNOLOGY CO LTD | GENER...,NGF small interfering nucleic acid molecule an...,The invention relates to a double stranded rib...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),False,NaN,"A61P17/06, C12N15/1136",073914809
